# Interrogación 1 - Métricas de Recomendación



## Instrucciones Generales

1. Está prohibido el uso de IA como chat GPT, Gemini o similares. Si se sorprende a alguien usándolo se le pedirá la prueba y se evaluará con 1,1.
2. Pueden realizar el práctico tanto en Google Colab como en un Jupyter Notebook local, pero no olvide entregar antes de que termine la hora de clases en este buzón. Debe entregar únicamente el archivo main.ipynb completo.
3. Esta actividad es individual, está prohibido conversar con compañeros para realizarla.
4. Sí está permitido el uso de código de los prácticos realizados en las ayudantías, tanto el que se te entrega por el equipo docente como el que tú mismo/a has realizado.

Para realizar este práctico solo es necesario tener instalado Numpy. Se sugiere tener instalada la última versión de Python y de Numpy.

La evaluación constará de una parte teórica y una segunda parte práctica donde deberás implementar con código en Python la métrica solicitada.

**NOMBRE ESTUDIANTE**: < colocar_nombre >

## Preguntas Teóricas (3pts)

En esta sección deberás contestar 3 preguntas teóricas

1. Explique brevemente **cómo se calculan** las métricas MAP (Mean Average Precision) y AUC (ROC Area Under the Curve). Luego, mencione en que se diferencian ambas métricas, especialmente en cuanto a la interpretación de sus resultados.

**Respuesta**: -Coloque su respuesta en esta celda-

2. Explique brevemente la **intuición principal en la métrica nDCG** (normalized Discounted Cummulative Gain). ¿Qué se requiere para poder calcularla? ¿Qué diferencia tiene con DCG? ¿Qué fortalezas tiene nDCG en relación a otras métricas de ranking vistas en el curso?

**Respuesta**: -Coloque su respuesta en esta celda-

3. Explica con tus palabras cuál es la diferencia entre RMSE y MAE, **tanto en la forma en que se calculan como en la forma de interpretar su resultado**. ¿En qué situaciones podría ser más apropiado usar cada una para evaluar un modelo de recomendación?

**Respuesta:** -Coloque su respuesta en esta celda-

## Implementeación de Métrica (7 pts)

En esta sección deberás implementar la métrica AUC (ROC Area Under the Curve), en base a la definción de average AUC del artículo BPR de Rendle et al. (seccion 4.1.1 en https://arxiv.org/pdf/1205.2618). Para hacer la evaluación de esta métrica vamos a utilizar el algoritmo de *collaborative filtering* denominado **Sapling Similarity**, presentado en el paper [Sapling Similarity: a performing and interpretable memory-based tool for recommendation](https://arxiv.org/abs/2210.07039), aplicado al dataset MovieLens 100K. Este dataset contiene 100.000 tuplas (usuario, item, ranking), de las cuales 80.000 se utilizarán para entrenamiento y 20.000 para testear. Los rankings se encuentran en un rango de 1 a 5.

Contarán con dos instancias del modelo ya entrenados: Basada en Usuario y Basada en Items. No deben modificar el código del modelo ni del entrenamiento.

In [ ]:
from sapling_similarity import SaplingSimilarity
import numpy as np

In [ ]:
# modelo basado en usuarios
sapling_similarity_users_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="user"
)

sapling_similarity_users_model.train()

# modelo basado en items
sapling_similarity_items_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="item"
)

sapling_similarity_items_model.train()

Los modelos entrenados cuenta con el método `test`, el cual retorna dos elementos:
- El primer elemento corresponde a una lista de tuplas con los **ratings reales** para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_real).

- El segundo elemento corresponde a una lista de tuplas con los **ratings predichos** por el algoritmo para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_predicted).

In [ ]:
y_test, y_pred_users = sapling_similarity_users_model.test()
y_test, y_pred_items = sapling_similarity_items_model.test()

Además, los modelos entrenados cuentan con el método `predict`, que retorna para un `user_id` específico las lista de tuplas (item_id, ranking) de los items que el usuario no ha evaluado. Con estas predicciones es posible rankear una lista de items para un usuario ordenando descendente los items según el rating que predice el modelo.

In [ ]:
sapling_similarity_users_model.predict(user_id=1)[40:50]

Se les entrega un diccionario con las predicciones hechas por cada una de las versiones del algoritmo, donde cada llave es un user_id y tiene como valor una lista con tuplas (item, rating), donde rating es el valor que el modelo predice que el usuario asignará al item. Los elementos para cada usuario estan ordenadas por el rating que predijo el modelo.

In [ ]:
user_based_predictions = {user_id: sapling_similarity_users_model.predict(user_id) for user_id in sapling_similarity_users_model.user_map.keys()}
item_based_predictions = {user_id: sapling_similarity_items_model.predict(user_id) for user_id in sapling_similarity_items_model.user_map.keys()}

1. Crea un diccionario con los items relevantes para cada usuario presentes en `y_test`. Cada llave del diccionario será un user_id y tendrá como valor un lista que contenga los item_ids de los items relevantes para ese usuario. Considere como relevante solo los items que se les asignó un **rating mayor o igual a 3**.

Ejemplo:
```python
{
  1: [3, 4, 8,...],
  2: [23, 24, 53,...],
  ...
}
```
donde [3, 4, 8,...] son los items relevantes para el usuario con ID = 1.

In [ ]:
user_relevant_items = {}
# COMPLETAR


2. Completa la función `user_auc`, la cual recibe como parámetros:

- `user_recommendations`:, Lista de tuplas (item_id, rating_predicted) de un usuario.

- `user_relevant_items` Lista de items_ids relevantes para el usuario.

- `k`: Largo de la lista de recomendación para calcular AUC.

Esta función deberá retornar el AUC para un usuario en particular. Basate en la siguiente fórmula para implementarlo:

$$
\text{AUC} =  \frac{1}{|I_u| \cdot |I \backslash I_u |} \sum_{(u,i,j) \in D_p} \delta(\hat{\phi}_{ui} \succ \hat{\phi}_{uj})
$$
donde:
- $|I_u|$: cantidad de items relevantes para el usuario $u$.
- $|I \backslash I_u|$: cantidad de items no relevantes para el usuario $u$.
- $(u,i,j)$: tripleta de usuario $u$, item relevante $i$ e item no relevante $j$.
- $D_p$: dataset para evaluación.
- $\delta(\hat{\phi}_{ui} \succ \hat{\phi}_{uj})$: Esta función vale 1 si el modelo ordenó correctamente un item relevante $i$ por encima de un item no relevante $j$ (es decir, si la predicción de $i$ es mayor a la de $j$). En caso contrario, vale 0.

---

**Ejemplo**

Supongamos que para un usuario tenemos:

`user_relevant_items = [A, B]`

`user_recommendations = [(A, 4.5), (C, 4.4), (B, 3.9), (D, 2.6)]`

Al comparar las tuplas de items (relevante, no_relevante) quedaría:
- (A, C): 4.5 > 4.4 → δ = 1
- (A, D): 4.5 > 2.6 → δ = 1
- (B, C): 3.9 > 4.4 → δ = 0 (mal ordenado)
- (B, D): 3.9 > 2.6 → δ = 1

Se tendría para dicho usuario un AUC de:
$$
\sum \delta = 3, AUC = \frac{3}{4} = 0.75
$$

In [ ]:
def user_auc(user_recommendations, user_relevant_items, k):
    # COMPLETAR
    pass

3. Llama a la función `average_auc` para cada modelo y compare usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores.

In [ ]:
def average_auc(predictions_dict, k):
    """
    Calcula el AUC promedio dado un diccionario de predicciones y un tamaño k de recomendaciones.
    """
    total_auc_users = []

    for user_id, predictions in predictions_dict.items():
        if user_id in user_relevant_items:
            # Convertir tripletas (user_id, item_id, rating) a pares (item_id, rating)
            rec_list = []
            for _, item_id, rating in predictions:
                rec_list.append((int(item_id), float(rating)))

            auc_value = user_auc(rec_list, user_relevant_items[user_id], k)
            if auc_value > 0:
                total_auc_users.append(auc_value)

    return sum(total_auc_users) / len(total_auc_users) if total_auc_users else 0.0


In [ ]:
# COMPLETAR